# Day 2, Segment 2: Context Window Management & Compression

## Overview

LLMs have **token limits** (e.g., GPT-4o has 128k context window). When you retrieve many documents, you risk exceeding these limits. This segment teaches you to:

1. **Count tokens** in your prompts
2. **Compress context** intelligently using LLM summarization
3. **Manage context windows** dynamically
4. **Balance quality vs. token usage**

### The Challenge
As you add more retrieved documents to your prompt, token count grows. But:
- Exceeding token limit → API error
- Removing documents → Loss of information
- Naive truncation → Critical info lost

### The Solution: Smart Compression
Use an LLM to summarize context while preserving key insights. Trade inference speed for better information density.

### Pipeline Architecture
```
Raw Context (many tokens)
    ↓
Count Tokens
    ↓
Exceeds limit? → Compress via LLM → Maintain quality
    ↓
Use compressed context in final answer
```

Let's build this step-by-step!

In [22]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

print("✓ Environment configured")

✓ Environment configured


## Step 1: Import Dependencies

**What you need**:
- `ChatOpenAI`: For LLM inference (both main and compression)
- `tiktoken`: OpenAI's token counter (accurate for GPT-4o)
- `Document`: LangChain document class
- System/HumanMessage: For structured prompts

**Key concept**: Tiktoken uses BPE (Byte-Pair Encoding) to count exactly how many tokens a string will consume.

In [23]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.documents import Document
import tiktoken

print("✓ All dependencies imported")

✓ All dependencies imported


## Step 2: Token Counting Utility

**Goal**: Accurately count tokens in text using OpenAI's tokenizer.

**Why it matters**:
- LLMs charge by tokens, not characters
- Token count determines if prompt fits in context window
- Enables dynamic context window management

**How it works**:
1. Get the encoding for the model (e.g., "gpt-4o")
2. Encode the text to tokens
3. Count the resulting tokens

**Example**:
```
"Hello world" → ~2-3 tokens
"What is machine learning?" → ~5-7 tokens
(Exact count depends on tokenizer)
```

**Key insight**: You can't just divide characters by 4! Tokenization is non-linear.

In [24]:
def count_tokens(text, model="gpt-4o"):
    """
    Count tokens in text using OpenAI's tokenizer.
    
    Args:
        text: String to tokenize
        model: Model name (must match tokenizer)
        
    Returns:
        Number of tokens
    """
    encoding = tiktoken.encoding_for_model(model)
    return len(encoding.encode(text))


# Test: Count tokens in sample text
sample = "Subscription pricing works well for enterprise SaaS."
tokens = count_tokens(sample)
print(f"✓ Token counter ready")
print(f"  Sample text: '{sample}'")
print(f"  Token count: {tokens}")

✓ Token counter ready
  Sample text: 'Subscription pricing works well for enterprise SaaS.'
  Token count: 9


## Step 3: Sample Documents

**Context**: These are documents retrieved from your vector store about pricing strategies.

**Key insights** (what you want to preserve during compression):
- Subscription model → predictable revenue
- Freemium → fails in enterprise
- Tiered pricing → segments customers
- Enterprise → values reliability

**Challenge**: All 4 documents together may exceed token limit. We'll compress intelligently while keeping these insights.

In [ ]:
documents = [
    Document(page_content="Subscription pricing works well for enterprise SaaS because it ensures predictable revenue and aligns with long-term contracts."),
    Document(page_content="Freemium models often fail in enterprise due to security, compliance, and onboarding complexity."),
    Document(page_content="Tiered pricing helps segment customers and increase revenue by aligning features with willingness to pay."),
    Document(page_content="Enterprise customers prefer reliability, support, and predictable billing over low-cost options."),
]

print(f"✓ Loaded {len(documents)} sample documents")

## Step 4: Context Builder

**Goal**: Combine multiple documents into a single context string for the LLM.

**What to do**:
1. Extract `page_content` from each document
2. Join them with `"\n\n"` separator

**Why it matters**:
- Creates the "ground truth" string that will be fed to LLM
- Token count of this combined string determines if compression is needed

**Example**:
```
Input: [Doc1, Doc2, Doc3]
Output: "Doc1 content\n\nDoc2 content\n\nDoc3 content"
```

In [ ]:
def build_context(docs):
    """
    Combine multiple documents into a single context string.
    
    Args:
        docs: List of Document objects
        
    Returns:
        String with documents separated by double newlines
    """
    return "\n\n".join([doc.page_content for doc in docs])


# Test: Build context from documents
context = build_context(documents)
print(f"✓ Context built")
print(f"  Tokens: {count_tokens(context)}")
print(f"  Characters: {len(context)}")
print(f"\n--- Preview ---")
print(context[:200] + "...")

## Step 5: Context Compression (LLM Summarization)

**Goal**: Summarize verbose context while preserving critical insights.

**The Pattern**:
- Instead of truncating, use an LLM to intelligently compress
- Trade: 1 extra API call → better information density → higher quality answers

**Key decisions**:
- Use `gpt-4o-mini`: Cheaper for summarization tasks
- System message emphasizes "key insights" preservation
- No token limit on output (let LLM decide optimal length)

**Why this works**:
- LLMs understand semantic importance
- Removes redundancy better than keyword extraction
- Maintains context flow and reasoning

**Production considerations**:
- Compression adds latency (extra API call)
- Cost: mini models are 5-10x cheaper
- Use only when necessary (context > limit)

In [ ]:
def compress_context(context, max_tokens=200):
    """
    Compress context using LLM summarization.
    
    Args:
        context: Full context string
        max_tokens: Target token count (guide, not strict limit)
        
    Returns:
        Compressed context preserving key insights
    """
    llm = ChatOpenAI(
        model="gpt-4o-mini",  # Cheaper for summarization
        temperature=0,         # Deterministic summaries
        openai_api_key=os.getenv("OPENAI_API_KEY")
    )

    messages = [
        SystemMessage(content="Summarize the following context while preserving key insights. Be concise."),
        HumanMessage(content=context)
    ]

    response = llm.invoke(messages)
    return response.content


# Test: Compress context
print(f"Original context tokens: {count_tokens(context)}")
compressed = compress_context(context)
print(f"Compressed context tokens: {count_tokens(compressed)}")
print(f"\n--- Compressed version ---")
print(compressed)

## Step 6: Context Window Management (Core Pattern)

**Goal**: Dynamically manage context size, compressing only when necessary.

**Decision logic**:
```
Measure tokens in raw context
    ↓
Within limit? → Use as-is (no extra cost)
    ↓
Over limit? → Compress (trade API call for information density)
```

**Why this pattern**:
- Lazy compression: Only compress when needed
- Cost-effective: Most contexts fit naturally
- Quality-preserving: Uncompressed context is always better when possible

**Key parameters**:
- `max_tokens`: Your budget (e.g., 200 tokens)
- Depends on: Final prompt size + desired answer length

**Real-world example**:
- Full context: 500 tokens (too large)
- After compression: 120 tokens (fits in budget)
- Final prompt: 120 context + 50 query = 170 tokens
- Answer budget: 30 tokens remaining

In [ ]:
def manage_context_window(docs, max_tokens=200):
    """
    Manage context window: measure, compress if needed, return optimized context.
    
    Args:
        docs: List of Document objects
        max_tokens: Maximum allowed tokens in context
        
    Returns:
        Context string (compressed if necessary)
    """
    # Step 1: Build raw context
    context = build_context(docs)

    # Step 2: Measure tokens
    tokens = count_tokens(context)
    print(f"Original tokens: {tokens}")

    # Step 3: Decision logic
    if tokens <= max_tokens:
        # Within limit → return as-is
        print(f"✓ Within limit ({max_tokens}), no compression needed")
        return context

    print(f"✗ Exceeds limit ({max_tokens}) → compressing...")

    # Step 4: Compress
    compressed = compress_context(context, max_tokens)
    compressed_tokens = count_tokens(compressed)
    print(f"✓ Compressed tokens: {compressed_tokens}")

    return compressed


# Test: Manage context with strict limit
print("--- Test: max_tokens=200 ---")
optimized_context = manage_context_window(documents, max_tokens=200)

## Step 7: Integration with Answer Generation

**Goal**: Use managed context in the final LLM call to generate answers.

**Pattern**:
1. Accept raw documents
2. Optimize context (measure + compress if needed)
3. Generate answer using optimized context
4. Return clean answer to user

**Key insight**: Context optimization happens transparently. User doesn't need to worry about token limits!

**Prompt structure**:
```
System: "You are a product strategist."
User: "Context:
[OPTIMIZED_CONTEXT]

Question:
[QUERY]"
```

**Quality preservation**:
- Uncompressed context used when possible (higher quality)
- Compression only when necessary (better than truncation)
- System message + query remain unchanged

In [ ]:
def generate_answer(query, docs):
    """
    Generate answer with optimized context (measured and compressed if needed).
    
    Args:
        query: User question
        docs: List of Document objects
        
    Returns:
        Generated answer grounded in context
    """
    # Step 1: Optimize context
    context = manage_context_window(docs, max_tokens=150)

    # Step 2: Initialize LLM (full-power model for generation)
    llm = ChatOpenAI(
        model="gpt-4o",
        temperature=0,
        openai_api_key=os.getenv("OPENAI_API_KEY")
    )

    # Step 3: Build prompt
    messages = [
        SystemMessage(content="You are a product strategist."),
        HumanMessage(content=f"""
Context:
{context}

Question:
{query}
""")
    ]

    # Step 4: Generate answer
    return llm.invoke(messages).content

## Step 8: Test the Complete Pipeline

**Scenario**: Product strategist asks about pricing strategy.

**What will happen**:
1. Raw documents are loaded
2. Context built and measured
3. If too large → LLM compresses intelligently
4. Compressed context used in final prompt
5. Answer generated

**Expected output**: Pricing recommendation grounded in context

Run the test below:

In [ ]:
query = "What pricing strategy should we use for enterprise customers?"

print(f"📝 Query: {query}\n")

response = generate_answer(query, documents)

print(f"\n💡 Answer:\n{response}")

Original tokens: 70
Within limit, no compression needed
For enterprise customers, a subscription-based pricing strategy is generally the most effective approach. This strategy aligns well with the needs and preferences of enterprise clients for several reasons:

1. **Predictable Revenue**: Subscription pricing ensures a steady and predictable revenue stream, which is crucial for financial planning and stability. Enterprises often have long-term budgets and prefer predictable expenses.

2. **Long-term Contracts**: Enterprises typically engage in long-term contracts, and subscription models naturally align with this preference. This can lead to higher customer retention and reduced churn.

3. **Reliability and Support**: Enterprise customers value reliability and comprehensive support. A subscription model can include premium support services, ensuring that customers receive the assistance they need, which can be a significant differentiator.

4. **Tiered Pricing**: Implementing a tiered

---

## Key Takeaways

### 1. **Token Management is Critical**
- LLMs have context limits (despite being large)
- Must count tokens to know your constraints
- Different models have different tokenizers

### 2. **Smart Compression Over Truncation**
- Naive truncation loses important info
- LLM compression preserves meaning
- Trade: 1 extra API call → better information density

### 3. **Lazy Compression Pattern**
- Only compress when necessary
- Uncompressed context is always better
- Reduces API calls and latency when context fits

### 4. **Component Separation**
- Token counting: Utility function
- Compression: Dedicated function (can swap strategies)
- Context management: Decision logic
- Answer generation: Final LLM call

### 5. **Cost-Quality Tradeoff**
| Strategy | Cost | Quality | Latency |
|----------|------|---------|---------|
| No limit (danger) | High (exceeds token limit) | ✗ Error | N/A |
| Truncation | Low | ⚠️ Medium (lost info) | Fast |
| Lazy compression | Medium | ✅ High (smart compression) | Slower |
| Extract-only | Low | ⚠️ Medium (rigid) | Fast |

---

## Production Patterns

### Pattern 1: Token Budget Calculation
```
Total budget: 128k tokens
System prompt: 100 tokens
Context: ? (dynamic)
User query: 50 tokens
Answer: 500 tokens (reserved)

Available for context = 128k - 100 - 50 - 500 = ~127k
```

### Pattern 2: Graceful Degradation
```python
def manage_context_progressive(docs, hard_limit=128000):
    context = build_context(docs)
    if count_tokens(context) <= hard_limit:
        return context
    # Fallback 1: Compress
    compressed = compress_context(context)
    if count_tokens(compressed) <= hard_limit:
        return compressed
    # Fallback 2: Take top-k docs only
    return build_context(docs[:3])
```

### Pattern 3: Adaptive Compression
```python
# Compress more aggressively if needed
compression_level = "Light" if tokens < limit else "Aggressive"
```

---

## Real-World Scenarios

### Scenario 1: Small Context (typical)
- Raw context: 150 tokens
- Limit: 200 tokens
- Outcome: No compression, fast execution

### Scenario 2: Large Context (needs compression)
- Raw context: 2000 tokens
- Limit: 200 tokens
- Compression: ~60% reduction needed
- Outcome: Extra API call, preserved meaning

### Scenario 3: Massive Context (fallback needed)
- Raw context: 50k tokens
- Limit: 200 tokens
- Fallback: Take top-k documents only
- Outcome: Graceful degradation

---

## Debugging Tips

### Check token counts at each step:
```python
raw = build_context(docs)
print(f"Raw: {count_tokens(raw)} tokens")

compressed = compress_context(raw)
print(f"Compressed: {count_tokens(compressed)} tokens")

reduction = (count_tokens(raw) - count_tokens(compressed)) / count_tokens(raw)
print(f"Reduction: {reduction:.1%}")
```

### Verify compression quality:
```python
# Manual check: Does compressed still have key info?
key_terms = ["pricing", "enterprise", "subscription"]
for term in key_terms:
    if term not in compressed.lower():
        print(f"⚠️ Warning: '{term}' lost in compression")
```

### Monitor token costs:
```python
# Track compression API calls for cost
compression_calls = 0
total_compression_tokens = 0
```

---

## Next Steps

In Segment 3, you'll learn:
- Multi-query expansion (retrieve with variations)
- Metadata filtering (reduce context upfront)
- Dynamic context window sizing
- Hybrid compression strategies

You now have the tools to handle real-world RAG at scale! 🚀